In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2023.csv")

In [3]:
df.shape

(127132, 34)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127132 entries, 0 to 127131
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          127132 non-null  object 
 1   order_date              127132 non-null  object 
 2   customer_id             127132 non-null  object 
 3   product_id              127132 non-null  object 
 4   product_name            127132 non-null  object 
 5   category                127132 non-null  object 
 6   subcategory             127132 non-null  object 
 7   brand                   127132 non-null  object 
 8   original_price_inr      127132 non-null  object 
 9   discount_percent        127132 non-null  float64
 10  discounted_price_inr    127132 non-null  float64
 11  quantity                127132 non-null  int64  
 12  subtotal_inr            127132 non-null  float64
 13  delivery_charges        116970 non-null  float64
 14  final_amount_inr    

In [5]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [6]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [7]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


C:\Users\hp\AppData\Local\Temp\ipykernel_15000\570715499.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


In [8]:
dfc['order_date'].head(20)

0     2023-01-16
1     2023-01-15
2     2023-01-10
3     2023-01-04
4     2023-01-07
5            NaN
6     2023-01-29
7     2023-01-08
8     2023-01-03
9     2023-01-05
10    2023-01-27
11    2023-01-08
12    2023-01-25
13    2023-01-26
14    2023-01-24
15    2023-01-06
16    2023-01-13
17    2023-01-09
18    2023-01-11
19           NaN
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [9]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [10]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [11]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [13]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [14]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [15]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [16]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [17]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [ ]:
print("Rows deleted:", (df_deduped))

In [19]:
len(dfc)

127132

In [20]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [21]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [22]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [23]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [24]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [25]:
dfc["payment_method"].unique()

array(['UPI', 'Credit Card', 'Net Banking', 'Debit Card',
       'Cash on Delivery', 'BNPL', 'WALLET'], dtype=object)

In [26]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2023_00000001', 'TXN_2023_00000002', 'TXN_2023_00000003', 'TXN_2023_00000004', 'TXN_2023_00000005', 'TXN_2023_00000006', 'TXN_2023_00000007', 'TXN_2023_00000008', 'TXN_2023_00000009', 'TXN_2023_00000010', 'TXN_2023_00000011', 'TXN_2023_00000012', 'TXN_2023_00000013', 'TXN_2023_00000014', 'TXN_2023_00000015', 'TXN_2023_00000016', 'TXN_2023_00000017', 'TXN_2023_00000018', 'TXN_2023_00000019', 'TXN_2023_00000020', 'TXN_2023_00000021', 'TXN_2023_00000022', 'TXN_2023_00000023', 'TXN_2023_00000024', 'TXN_2023_00000025', 'TXN_2023_00000026', 'TXN_2023_00000027', 'TXN_2023_00000028', 'TXN_2023_00000029', 'TXN_2023_00000030', 'TXN_2023_00000031', 'TXN_2023_00000032', 'TXN_2023_00000033', 'TXN_2023_00000034', 'TXN_2023_00000035', 'TXN_2023_00000036', 'TXN_2023_00000037', 'TXN_2023_00000038', 'TXN_2023_00000039', 'TXN_2023_00000040', 'TXN_2023_00000041', 'TXN_2023_00000042', 'TXN_2023_00000043', 'TXN_2023_00000044', 'TXN_2023_00000045', 'TXN_2

In [27]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0', '40.0']

final_amount_inr: ['10000.94', '100007.18', '100008.91', '100009.05', '100012.85', '10002.53', '10002.85', '100021.34', '100028.3', '100032.94', '10004.25', '10004.51', '10004.76', '100042.56', '100044.18', '100045.78', '100049.36', '100052.37', '100055.26', '10006.88', '10006.93', '100069.46', '10007.14', '10007.45', '100077.29999999999', '10008.13', '10008.6', '100099.48', '10010.13', '100101.87', '100107.94', '100109.16', '10011.94', '100110.82', '100116.64', '100127.82', '10013.65', '10014.08', '10014.8', '100141.44', '100146.04', '100148.42', '100148.53', '10015.94', '10016.63', '100165.25', '100167.16', '100169.01', '10017.54', '100170.14', '100170.43', '100178.24', '10018.36', '10018.46', '10018.79', '100183.09', '10019.49', '10021.39', '10021.64', '10021.65', '10022.04', '10022.65', '10022.86', '100229.12', '10023.42', '10023.9', '100232.07', '100235.32', '100245.49', '100245.72', '10025.4', '10025.79', '10027.6

In [28]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2023']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.27', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.4', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.7', '0.71', '0.72', '0.73', '0.75', '0.76', '0.78', '1.2', '1.21', '1.24', '1.27', '1.29', '1.31', '1.32', '1.33', '1.37', '1.39', '1.4', '1.42', '1.46', '1.48', '1.5', '1.51', '1.55',

In [29]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [30]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [32]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [34]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2023_clean.csv",header='infer',index=False)

In [33]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2023_00000001,2023-01-16,CUST_2023_00005822,PROD_001899,Xiaomi Watch Premium,Electronics,Smart Watch,Xiaomi,59416.51,0.00,...,False,NaN,4.0,Returned,1,2023,1,0.06,True,4.2
1,TXN_2023_00000002,2023-01-15,CUST_2019_00019403,PROD_000472,Oppo R17 Pro 64GB Black,Electronics,Smartphones,Oppo,19474.76,0.00,...,False,NaN,4.0,Returned,1,2023,1,0.16,True,3.4
2,TXN_2023_00000003,2023-01-10,CUST_2023_00040022,PROD_001759,JBL Neckband,Electronics,Audio,JBL,27097.23,27.71,...,False,NaN,5.0,Delivered,1,2023,1,0.23,True,3.7
3,TXN_2023_00000004,2023-01-04,CUST_2023_00021818,PROD_001158,Xiaomi Redmi Note 12 256GB White,Electronics,Smartphones,Xiaomi,12883.85,0.00,...,False,NaN,3.5,Delivered,1,2023,1,0.23,True,4.1
4,TXN_2023_00000005,2023-01-07,CUST_2020_00048935,PROD_000221,Apple iPhone 8 32GB Blue,Electronics,Smartphones,Apple,165637.56,0.00,...,False,NaN,3.0,Delivered,1,2023,1,0.22,False,3.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127127,TXN_2023_00010663_DUP,2023-02-10,CUST_2021_00032315,PROD_001683,Apple Mi Pad 8GB RAM Silver,Electronics,Tablets,Apple,90033.17,65.49,...,True,Valentine Sale,NaN,Delivered,2,2023,1,0.40,True,4.5
127128,TXN_2023_00082508_DUP,2023-09-29,CUST_2022_00015469,PROD_000278,Xiaomi Redmi 4A 16GB White,Electronics,Smartphones,Xiaomi,22108.70,31.35,...,NaN,Amazon Great Indian Festival,5.0,Delivered,9,2023,3,0.21,True,4.1
127129,TXN_2023_00023903_DUP,2023-03-08,CUST_2019_00016252,PROD_001014,OnePlus OnePlus 10T 64GB White,Electronics,Smartphones,OnePlus,58697.10,23.90,...,True,Holi Festival,5.0,Delivered,3,2023,1,0.19,NaN,4.5
127130,TXN_2023_00093042_DUP,2023-10-04,CUST_2022_00015673,PROD_000195,Motorola Moto G4 32GB Blue,Electronics,Smartphones,Motorola,15514.86,33.62,...,True,Amazon Great Indian Festival,3.0,Delivered,10,2023,4,0.15,NaN,4.2
